# Experimental - Find All Groups of a Given Order

In [1]:
import itertools as it
import finalg as fg

Finite Algebras:
  To see a listing of built-in example algebras, run finalg.examples.about()
  To retrieve an example algebra, use finalg.examples[INDEX]



The code below is does a brute-force search for all valid Cayley tables for a given group order.

In particular, the function `generate_all_group_tables` returns a list of all possible Cayley tables for a given order, n.

The function `tables_to_groups` uses the output list from `generate_all_group_tables` to create a list of `finalg` Groups based on the list of tables.

So, for example, for n=3, there is only one possible table/group.

For n=4 there are four valid tables/groups, the first of which is not isomorphic to any of the others, but the last three are all isomorphic to each other. So, up to isomorphism, there are only two groups of order 4, as expected.

We know that there is only one valid group of order 5. The function `generate_all_group_tables` generates 6 tables/groups for n=5, but all are isomorphic to each other, again, as expected.

In [31]:
# ---------------------------------
#         EXPERIMENTAL
# Find all groups of a given order
# ---------------------------------

def _no_conflict(p1, p2):
    """Returns True only if no element of p1 equals the corresponding element of p2."""
    return all([p1[i] != p2[i] for i in range(len(p1))])


def _no_conflicts(items):
    """Return True if each possible pair, from a list of items, has no conflicts."""
    return all(_no_conflict(combo[0], combo[1]) for combo in it.combinations(items, 2))


def _filter_out_conflicts(perms, perm, n):
    """Filter out all permutations in perms that conflict with perm,
    and don't have n as the first element."""
    nperms = [q for q in perms if q[0] == n]
    return [p for p in nperms if _no_conflict(p, perm)]


def generate_all_group_tables(order):
    """Experimental Code: Return a list of all arrays that correspond to multiplication tables for groups
    of a specific order.

    WARNING: The algorithm here is inefficient, so even very small values of 'order' will result in very
    long runtimes (e.g., order=6 ==> ~5-6 hrs runtime).
    """
    row0 = list(range(order))
    row_candidates = [[row0]]
    for row_num in range(1, order):
        row_candidates.append(_filter_out_conflicts(it.permutations(row0), row0, row_num))
    table_candidates = [cand for cand in it.product(*row_candidates) if is_table_associative(list(cand))]  # ***
    return [tbl for tbl in table_candidates if _no_conflicts(tbl)]


# TODO: Reconcile this version with the similar method in CayleyTable.
def is_table_associative(table):
    """Determine whether the table supports an associative operation."""
    elements = table[0]  # The first row should correspond to the elements of a group
    for a in elements:
        for b in elements:
            for c in elements:
                ab = table[a][b]
                bc = table[b][c]
                if not (table[ab][c] == table[a][bc]):
                    return False
    return True


def tables_to_groups(tables, identity_name="e", elem_name="a"):
    """Given a list of multiplication tables, all the same size, turn them into a list of groups."""
    order = len(tables[0])
    groups = []
    for j in range(len(tables)):
        gname = f"G{j}"
        desc = f"Group {j} of order {order}"
        ename = elem_name + str(j) + "_"
        elements = [identity_name + str(j)] + [f"{ename}" + str(i) for i in range(1, order)]
        groups.append(fg.make_finite_algebra(gname, desc, elements, tables[j]))
    return groups

In [32]:
tables3 = generate_all_group_tables(3)
tables3

[([0, 1, 2], (1, 2, 0), (2, 0, 1))]

In [33]:
tables4 = generate_all_group_tables(4)
tables4

[([0, 1, 2, 3], (1, 0, 3, 2), (2, 3, 0, 1), (3, 2, 1, 0)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 3, 1, 0), (3, 2, 0, 1)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 3, 0, 1), (3, 0, 1, 2)),
 ([0, 1, 2, 3], (1, 3, 0, 2), (2, 0, 3, 1), (3, 2, 1, 0))]

In [34]:
z3 = tables_to_groups(tables3)[0]
z3

Group(
'G0',
'Group 0 of order 3',
('e0', 'a0_1', 'a0_2'),
[[0, 1, 2], [1, 2, 0], [2, 0, 1]]
)

In [35]:
z3.about()


** Group **
Name: G0
Instance ID: 5098673392
Description: Group 0 of order 3
Order: 3
Identity: 'e0'
Commutative? Yes
Cyclic?: Yes
Generators: ['a0_1', 'a0_2']
Elements:
   Index   Name   Inverse  Order
      0    'e0'    'e0'       1
      1  'a0_1'  'a0_2'       3
      2  'a0_2'  'a0_1'       3
Cayley Table (showing indices):
[[0, 1, 2], [1, 2, 0], [2, 0, 1]]


'<Group:G0, ID:5098673392>'

In [36]:
groups4 = tables_to_groups(tables4)

grp = groups4[0]
for g in groups4[1:]:
    print(f"{grp.name} isomorphic to {g.name}? {grp.fast_isomorphic(g)}")

G0 isomorphic to G1? False
G0 isomorphic to G2? False
G0 isomorphic to G3? False


In [37]:
grp = groups4[1]
for g in groups4[2:]:
    print(f"{grp.name} isomorphic to {g.name}? {grp.fast_isomorphic(g)}")

G1 isomorphic to G2? {'a1_2': 'a2_1', 'e1': 'e2', 'a1_1': 'a2_2', 'a1_3': 'a2_3'}
G1 isomorphic to G3? {'a1_2': 'a3_1', 'e1': 'e3', 'a1_1': 'a3_3', 'a1_3': 'a3_2'}


In [38]:
tables5 = generate_all_group_tables(5)
len(tables5)

6

In [39]:
groups5 = tables_to_groups(tables5)

grp = groups5[0]
for g in groups5[1:]:
    print(f"{grp.name} isomorphic to {g.name}? {grp.fast_isomorphic(g)}")

G0 isomorphic to G1? {'a0_1': 'a1_1', 'e0': 'e1', 'a0_2': 'a1_2', 'a0_3': 'a1_4', 'a0_4': 'a1_3'}
G0 isomorphic to G2? {'a0_1': 'a2_1', 'e0': 'e2', 'a0_2': 'a2_3', 'a0_3': 'a2_4', 'a0_4': 'a2_2'}
G0 isomorphic to G3? {'a0_1': 'a3_1', 'e0': 'e3', 'a0_2': 'a3_3', 'a0_3': 'a3_2', 'a0_4': 'a3_4'}
G0 isomorphic to G4? {'a0_1': 'a4_1', 'e0': 'e4', 'a0_2': 'a4_4', 'a0_3': 'a4_3', 'a0_4': 'a4_2'}
G0 isomorphic to G5? {'a0_1': 'a5_1', 'e0': 'e5', 'a0_2': 'a5_4', 'a0_3': 'a5_2', 'a0_4': 'a5_3'}


In [45]:
order = 4

In [46]:
row0 = list(range(order))
row0

[0, 1, 2, 3]

In [47]:
row_candidates = [[row0]]
row_candidates

[[[0, 1, 2, 3]]]

In [48]:
for row_num in range(1, order):
    row_candidates.append(_filter_out_conflicts(it.permutations(row0), row0, row_num))

row_candidates

[[[0, 1, 2, 3]],
 [(1, 0, 3, 2), (1, 2, 3, 0), (1, 3, 0, 2)],
 [(2, 0, 3, 1), (2, 3, 0, 1), (2, 3, 1, 0)],
 [(3, 0, 1, 2), (3, 2, 0, 1), (3, 2, 1, 0)]]

In [49]:
list(it.product(*row_candidates))

[([0, 1, 2, 3], (1, 0, 3, 2), (2, 0, 3, 1), (3, 0, 1, 2)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 0, 3, 1), (3, 2, 0, 1)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 0, 3, 1), (3, 2, 1, 0)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 3, 0, 1), (3, 0, 1, 2)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 3, 0, 1), (3, 2, 0, 1)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 3, 0, 1), (3, 2, 1, 0)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 3, 1, 0), (3, 0, 1, 2)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 3, 1, 0), (3, 2, 0, 1)),
 ([0, 1, 2, 3], (1, 0, 3, 2), (2, 3, 1, 0), (3, 2, 1, 0)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 0, 3, 1), (3, 0, 1, 2)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 0, 3, 1), (3, 2, 0, 1)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 0, 3, 1), (3, 2, 1, 0)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 3, 0, 1), (3, 0, 1, 2)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 3, 0, 1), (3, 2, 0, 1)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 3, 0, 1), (3, 2, 1, 0)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 3, 1, 0), (3, 0, 1, 2)),
 ([0, 1, 2, 3], (1, 2, 3, 0), (2, 3, 1, 0), (3, 2, 0, 1)

In [50]:
help(it.product)

Help on class product in module itertools:

class product(builtins.object)
 |  product(*iterables, repeat=1)
 |
 |  Cartesian product of input iterables.  Equivalent to nested for-loops.
 |
 |  For example, product(A, B) returns the same as:  ((x,y) for x in A for y in B).
 |  The leftmost iterators are in the outermost for-loop, so the output tuples
 |  cycle in a manner similar to an odometer (with the rightmost element changing
 |  on every iteration).
 |
 |  To compute the product of an iterable with itself, specify the number
 |  of repetitions with the optional repeat keyword argument. For example,
 |  product(A, repeat=4) means the same as product(A, A, A, A).
 |
 |  product('ab', range(3)) --> ('a',0) ('a',1) ('a',2) ('b',0) ('b',1) ('b',2)
 |  product((0,1), (0,1), (0,1)) --> (0,0,0) (0,0,1) (0,1,0) (0,1,1) (1,0,0) ...
 |
 |  Methods defined here:
 |
 |  __getattribute__(self, name, /)
 |      Return getattr(self, name).
 |
 |  __iter__(self, /)
 |      Implement iter(self).
 

In [52]:
fg.Cayley_Table(list(([0, 1, 2, 3], (1, 0, 3, 2), (2, 0, 3, 1), (3, 0, 1, 2))))

AttributeError: module 'finalg' has no attribute 'Cayley_Table'

In [ ]:
table_candidates = [cand for cand in it.product(*row_candidates) if is_table_associative(list(cand))]  # ***